# MemeDecodeR v2
v1 (MAF fine-tune) overfit hard: train 97% vs val 74% vs **shifted test 58% (macro-F1 0.56)**. v2 attacks the shift gap instead of fitting harder:
- frozen CLIP-L/14@336 + SigLIP image features, multilingual-e5 text features (no full fine-tuning)
- shift-simulating augmentation (crop, colour, downscale, blur, JPEG, borders) on training images
- TF-IDF char/word n-gram model on the OCR text, blended with the multimodal head
- 5-fold CV, then a score on the labelled `testing_set` as the shift check, then the submission

Run top to bottom. Features are cached in `/kaggle/working/feats`.

## 1. GPU + installs

In [ ]:
!nvidia-smi -L
!pip install -q sentencepiece ftfy regex easyocr
!pip install -q git+https://github.com/openai/CLIP.git

import torch
assert torch.cuda.is_available(), 'No GPU: Settings -> Accelerator -> GPU T4 x2'

## 2. Data (labelled splits + competition test, OCR if needed)

In [ ]:
import os, io, re, glob, json, random, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
dev = 'cuda'

CLASSES = ['Politics', 'Genders', 'Religion', 'Neutral']          # submission label names
LABEL_MAP = {'political aggression': 'Politics', 'gendered aggression': 'Genders',
             'religious aggression': 'Religion', 'non-aggressive': 'Neutral'}
N_AUG = 3                       # augmented "shifted" views per image (train-time only)
WORK = '/kaggle/working'; os.makedirs(f'{WORK}/feats', exist_ok=True)
IMG_EXT = ('.jpg', '.jpeg', '.png', '.webp')

# ---- locate the labelled dataset (same mount as v1) ----
DS_ROOT = '/kaggle/input/datasets/faiyazabrar/memedecoder-2'
DATASET = next((os.path.dirname(r) for r, _, _ in os.walk(DS_ROOT) if r.endswith(os.path.join('Dataset', 'Img'))), None)
assert DATASET, 'Dataset/Img not found under ' + DS_ROOT
IMG_DIR = f'{DATASET}/Img'

def resolve(n):
    p = f'{IMG_DIR}/{n}'
    if os.path.exists(p): return p
    for e in IMG_EXT:
        if os.path.exists(p + e): return p + e
    raise FileNotFoundError(n)

def norm_label(x):
    s = str(x).strip().lower()
    if s in LABEL_MAP: return LABEL_MAP[s]
    for c in CLASSES:
        if s.startswith(c.lower()[:4]): return c
    raise ValueError(f'unknown label {x!r}')

def clean(s): return re.sub(r'\s+', ' ', str(s)).strip()

def detect_cols(d):
    obj = [c for c in d.columns if not pd.api.types.is_numeric_dtype(d[c])]
    def hits(c):
        ok = 0
        for v in d[c].astype(str).head(50):
            try: resolve(v); ok += 1
            except FileNotFoundError: pass
        return ok
    ic = max(obj, key=hits)
    lc = 'Label' if 'Label' in d.columns else next(c for c in obj if c != ic and d[c].nunique() <= 6)
    tcs = [c for c in obj if c not in (ic, lc)]
    tc = max(tcs, key=lambda c: d[c].astype(str).str.len().mean()) if tcs else None
    return ic, lc, tc

parts = []
for split, f in [('train', 'training_set'), ('val', 'validation_set'), ('test', 'testing_set')]:
    d = pd.read_csv(f'{DATASET}/{f}.csv')
    ic, lc, tc = detect_cols(d)
    print(f'{f:<15} img={ic!r} label={lc!r} text={tc!r} rows={len(d)}')
    parts.append(pd.DataFrame({
        'name': d[ic].astype(str).values,
        'text': (d[tc].fillna('').map(clean).values if tc else ''),
        'y': d[lc].map(norm_label).values, 'split': split}))
df = pd.concat(parts, ignore_index=True)
df['path'] = df.name.map(resolve)

# ---- competition test set (auto-detected: biggest image folder outside the labelled dataset) ----
best = None
for r, _, fs in os.walk('/kaggle/input'):
    if r.startswith(DS_ROOT): continue
    imgs = sorted(f for f in fs if f.lower().endswith(IMG_EXT))
    if len(imgs) >= 50 and (best is None or len(imgs) > len(best[1])): best = (r, imgs)

SAMPLE_SUB = next(iter(glob.glob('/kaggle/input/**/*ample*ubmission*.csv', recursive=True)), None)
if best:
    r, imgs = best
    if SAMPLE_SUB:
        want = pd.read_csv(SAMPLE_SUB)['Image_name'].astype(str).tolist()
        imgs = [n for n in want if os.path.exists(f'{r}/{n}')] or imgs
    paths = [f'{r}/{n}' for n in imgs]
    import easyocr
    rd = easyocr.Reader(['bn', 'en'], gpu=True, verbose=False)
    ocr = [clean(' '.join(rd.readtext(p, detail=0, paragraph=True))) for p in paths]
    del rd; torch.cuda.empty_cache()
    df = pd.concat([df, pd.DataFrame({'name': imgs, 'text': ocr, 'y': None, 'split': 'comp', 'path': paths})], ignore_index=True)
    print(f'competition test: {len(imgs)} images from {r}  | sample_submission: {SAMPLE_SUB}')
else:
    print('!! no competition test folder found -> final cell falls back to the local testing_set')

y_all = df.y.map({c: i for i, c in enumerate(CLASSES)}).fillna(-1).astype(int).values
print(df.split.value_counts().to_dict(), '| empty text:', int((df.text == '').sum()))
print(df.groupby('split').y.value_counts(normalize=True).round(2).unstack().fillna(0))

## 3. Frozen features with shift augmentation

In [ ]:
# Frozen backbones + shift-simulating augmentation (cached to /kaggle/working/feats)
import torchvision.transforms as T
from PIL import Image, ImageFilter, ImageOps
from torch.utils.data import Dataset, DataLoader

def shift_aug(img, seed):
    """Cheap simulation of distribution shift: crop/zoom, colour, resolution loss, blur, JPEG, borders."""
    rng = random.Random(seed)
    w, h = img.size
    cw, ch = int(w * rng.uniform(0.75, 1.0)), int(h * rng.uniform(0.75, 1.0))
    x0, y0 = rng.randint(0, w - cw), rng.randint(0, h - ch)
    img = img.crop((x0, y0, x0 + cw, y0 + ch))
    img = T.ColorJitter(0.4, 0.4, 0.4, 0.05)(img)
    if rng.random() < 0.25: img = T.Grayscale(3)(img)
    if rng.random() < 0.5:
        f = rng.uniform(0.35, 0.7)
        img = img.resize((max(32, int(img.width * f)), max(32, int(img.height * f))), Image.BILINEAR)
    if rng.random() < 0.3: img = img.filter(ImageFilter.GaussianBlur(rng.uniform(0.5, 1.5)))
    if rng.random() < 0.6:
        buf = io.BytesIO(); img.save(buf, 'JPEG', quality=rng.randint(25, 70)); img = Image.open(buf).convert('RGB')
    if rng.random() < 0.3:
        img = ImageOps.expand(img, rng.randint(8, 40), fill=tuple(rng.randint(0, 255) for _ in range(3)))
    return img

class ImgDS(Dataset):
    def __init__(self, paths, pre, seed=None): self.p, self.pre, self.seed = list(paths), pre, seed
    def __len__(self): return len(self.p)
    def __getitem__(self, i):
        im = Image.open(self.p[i]).convert('RGB')
        if self.seed is not None: im = shift_aug(im, self.seed * 1_000_003 + i)
        return self.pre(im)

@torch.no_grad()
def extract(pre, enc, seed=None, bs=64):
    dl = DataLoader(ImgDS(df.path, pre, seed), batch_size=bs, num_workers=4, pin_memory=True)
    out = [torch.nn.functional.normalize(enc(x.to(dev).half()).float(), dim=-1).cpu().numpy() for x in dl]
    return np.concatenate(out)

def cached(name, fn):
    p = f'{WORK}/feats/{name}.npy'
    if os.path.exists(p): return np.load(p)
    a = fn(); np.save(p, a); return a

IMG = {}    # backbone -> (1+N_AUG, N, d); view 0 = clean
def run_backbone(name, pre, enc):
    IMG[name] = np.stack([cached(f'{name}_v{v}', lambda v=v: extract(pre, enc, None if v == 0 else v))
                          for v in range(N_AUG + 1)])
    print(name, IMG[name].shape)

# 1) CLIP ViT-L/14 @336
import clip
m, pre = clip.load('ViT-L/14@336px', device=dev); m.eval()
run_backbone('clip', pre, m.encode_image)
del m; torch.cuda.empty_cache()

# 2) SigLIP so400m (skipped automatically if it fails to load)
try:
    from transformers import AutoModel, AutoImageProcessor
    sid = 'google/siglip-so400m-patch14-384'
    sm = AutoModel.from_pretrained(sid).half().to(dev).eval()
    sp = AutoImageProcessor.from_pretrained(sid)
    def senc(x):
        o = sm.get_image_features(pixel_values=x)
        return o if torch.is_tensor(o) else o.pooler_output
    run_backbone('siglip', lambda im: sp(images=im, return_tensors='pt')['pixel_values'][0], senc)
    del sm; torch.cuda.empty_cache()
except Exception as e:
    print('SigLIP skipped:', repr(e)[:200])

# 3) Text: multilingual-e5-large on the OCR text (handles Bengali)
from transformers import AutoTokenizer, AutoModel
tk = AutoTokenizer.from_pretrained('intfloat/multilingual-e5-large')
tm = AutoModel.from_pretrained('intfloat/multilingual-e5-large').half().to(dev).eval()

@torch.no_grad()
def embed_text(texts, bs=64):
    out = []
    for i in range(0, len(texts), bs):
        b = tk(['query: ' + t for t in texts[i:i + bs]], padding=True, truncation=True, max_length=128, return_tensors='pt').to(dev)
        h = tm(**b).last_hidden_state
        m_ = b['attention_mask'].unsqueeze(-1).to(h.dtype)
        out.append(torch.nn.functional.normalize(((h * m_).sum(1) / m_.sum(1)).float(), dim=-1).cpu().numpy())
    return np.concatenate(out)

TXT = cached('e5', lambda: embed_text(df.text.tolist()))
del tm; torch.cuda.empty_cache()
print('text', TXT.shape)

## 4. Heads + 5-fold CV

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack

# --- text-only TF-IDF (char n-grams survive OCR noise; custom token pattern keeps Bengali matras) ---
tf_c = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), min_df=2, sublinear_tf=True, max_features=300_000)
tf_w = TfidfVectorizer(analyzer='word', ngram_range=(1, 2), min_df=2, sublinear_tf=True,
                       token_pattern=r'[^\s\.,!?;:"\'()\[\]{}\|/\\\-।]+')
TFX = hstack([tf_c.fit_transform(df.text), tf_w.fit_transform(df.text)]).tocsr()   # unsupervised fit on all text

def fit_tf(tr, C=10):
    return LogisticRegression(C=C, class_weight='balanced', max_iter=1000).fit(TFX[tr], y_all[tr])

# --- multimodal head on frozen features; clean + augmented views of each training image ---
def X_mm(idx, views):
    return np.vstack([np.hstack([IMG[k][v][idx] for k in IMG] + [TXT[idx]]) for v in views])

def fit_mm(tr, Cs):
    views = range(N_AUG + 1)
    X, y = X_mm(tr, views), np.tile(y_all[tr], len(views))
    sc = StandardScaler().fit(X); X = sc.transform(X)
    return sc, {C: LogisticRegression(C=C, class_weight='balanced', max_iter=300).fit(X, y) for C in Cs}

def pred_mm(model, idx):
    sc, clfs = model
    X = sc.transform(X_mm(idx, [0]))                    # clean view only at prediction time
    return {C: clf.predict_proba(X) for C, clf in clfs.items()}

mf1 = lambda y, p: f1_score(y, p.argmax(1), average='macro')

# --- 5-fold CV over train+val (augmented copies stay in the same fold as their source image) ---
lab = np.where(df.split.isin(['train', 'val']))[0]
Cs = [0.0005, 0.002, 0.01]
oof_mm = {C: np.zeros((len(df), 4)) for C in Cs}; oof_tf = np.zeros((len(df), 4))
for k, (a, b) in enumerate(StratifiedKFold(5, shuffle=True, random_state=SEED).split(lab, y_all[lab])):
    tr, va = lab[a], lab[b]
    for C, P in pred_mm(fit_mm(tr, Cs), va).items(): oof_mm[C][va] = P
    oof_tf[va] = fit_tf(tr).predict_proba(TFX[va])
    print(f'fold {k} done')

yl = y_all[lab]
for C in Cs: print(f'multimodal C={C:<7} OOF macroF1 {mf1(yl, oof_mm[C][lab]):.4f}')
print(f'tfidf-only          OOF macroF1 {mf1(yl, oof_tf[lab]):.4f}')
BEST_C = max(Cs, key=lambda C: mf1(yl, oof_mm[C][lab]))
ws = np.linspace(0, 1, 11)
sc_w = [mf1(yl, w * oof_mm[BEST_C][lab] + (1 - w) * oof_tf[lab]) for w in ws]
BEST_W = float(ws[int(np.argmax(sc_w))])
print(f'best C={BEST_C}, blend w(multimodal)={BEST_W:.1f} -> OOF macroF1 {max(sc_w):.4f}, acc {accuracy_score(yl, (BEST_W*oof_mm[BEST_C][lab]+(1-BEST_W)*oof_tf[lab]).argmax(1)):.4f}')

## 5. Shift check on labelled testing_set

In [ ]:
# The number that matters: train on train+val, score on the labelled (distribution-shifted) testing_set.
te = np.where(df.split == 'test')[0]
P = BEST_W * pred_mm(fit_mm(lab, [BEST_C]), te)[BEST_C] + (1 - BEST_W) * fit_tf(lab).predict_proba(TFX[te])
yt = y_all[te]
print(f'v2 testing_set: acc {accuracy_score(yt, P.argmax(1)):.4f} | macro-F1 {mf1(yt, P):.4f}   (v1: acc 0.5775 | macro-F1 0.5616)')
print(classification_report(yt, P.argmax(1), target_names=CLASSES, digits=3))
print(pd.DataFrame(confusion_matrix(yt, P.argmax(1)), index=CLASSES, columns=CLASSES))

## 6. Final fit + submission.csv

In [ ]:
# Final fit on ALL labelled data (train+val+testing_set: the testing_set adds shifted-domain examples), predict competition test.
comp = np.where(df.split == 'comp')[0]
if len(comp):
    fit_idx, out_idx = np.where(df.split.isin(['train', 'val', 'test']))[0], comp
else:
    print('!! competition test not found: predicting the local testing_set with a train+val model (rehearsal only)')
    fit_idx, out_idx = lab, te

P = BEST_W * pred_mm(fit_mm(fit_idx, [BEST_C]), out_idx)[BEST_C] + (1 - BEST_W) * fit_tf(fit_idx).predict_proba(TFX[out_idx])
sub = pd.DataFrame({'Image_name': df.name.values[out_idx], 'Target': [CLASSES[i] for i in P.argmax(1)]})

if SAMPLE_SUB and len(comp):
    ref = pd.read_csv(SAMPLE_SUB)['Image_name'].astype(str)
    assert set(ref) == set(sub.Image_name), f'name mismatch vs sample submission ({len(set(ref) ^ set(sub.Image_name))} differ)'
    sub = sub.set_index('Image_name').loc[ref].reset_index()

assert sub.Target.isin(CLASSES).all() and sub.Image_name.is_unique
sub.to_csv(f'{WORK}/submission.csv', index=False, encoding='utf-8')
print(sub.shape, sub.Target.value_counts().to_dict()); sub.head()